In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from jppype import vscode_theme

from fundus_vessels_toolkit.models.topology.dataset import BranchDigraphDataset
from fundus_vessels_toolkit.models.topology.losses import BranchDigraphMiner
from fundus_vessels_toolkit.models.topology.model import BranchDigraphModel
from fundus_vessels_toolkit.segment_to_graph.vbranch_digraph import VBranchDigraph
from train import DigraphGNNTrainer, DigraphGNNTrainerConfig

vscode_theme()


/home/gaby/.conda/envs/lab/lib/python3.13/site-packages/torch/__init__.py:1597: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:78.)
  _C._set_float32_matmul_precision(precision)


HTML(value="<style>\n        .cell-output-ipywidget-background {\n                background: transparent !imp…

In [3]:
dataset = BranchDigraphDataset("ALL_DATA_bundle.tar.gz")
train_set, val_set, test_set = dataset.split_sets(train_ratio=0.7, val_ratio=0.15)

Processing...
Done!


## Visualize result from pred table


In [4]:
def parse_arborescence(preds, idx, opti=False):
    pred = preds.loc[idx] if isinstance(idx, str) else preds.iloc[idx]
    b_parent = np.fromstring(pred[("opti_" if opti else "") + "parent"], sep=",", dtype=int)
    b_dir = np.fromstring(pred[("opti_" if opti else "") + "dir"], sep=",", dtype=int)
    b_av = np.fromstring(pred["av"], sep=",", dtype=int)
    return pred.name, b_parent, b_dir, b_av


### Load model from checkpoint


In [ ]:
# checkpoint = torch.load("GNN-Topo-v1/c2kx8j5h/checkpoints/epoch=239-step=8880.ckpt")
# checkpoint = torch.load("GNN-Topo-v1/ft6svpfg/checkpoints/epoch=179-step=3420.ckpt")
checkpoint = torch.load("GNN-Topo-v1/96dxz1ex/checkpoints/epoch=99-step=1900.ckpt")

model = BranchDigraphModel(checkpoint["hyper_parameters"]["config"]["model"])
model.load_state_dict({k[6:]: v for k, v in checkpoint["state_dict"].items() if k.startswith("model.")})
model = model.cuda().eval()

In [48]:
ID = 15
eval_set = test_set
sample_gt, gt_digraph = eval_set.get(ID, version="fvt", return_digraph=True)
with torch.inference_mode():
    out: BranchDigraphModel.Output = model(sample_gt.cuda())
pred_parent, pred_dir = out.max_parent(), out.dir_logit > 0
pred_parent, pred_dir, pred_av = out.optimal_tree

assert VBranchDigraph.has_fp_av_p(gt_digraph)
valid_branch = ~gt_digraph.branch_fp()
av_gt = gt_digraph.branch_av_class() <= 1
print(((out.av_logit.numpy(force=True)[valid_branch] > 0) == av_gt[valid_branch]).mean())
print(((pred_av.numpy(force=True)[valid_branch] > 0) == av_gt[valid_branch]).mean())


0.7892857142857143
0.6571428571428571


In [49]:
m, pred_tree = eval_set.show_tree_diff(
    out.name,
    pred_parent.numpy(force=True),
    pred_dir.numpy(force=True),
    out.fp_logit.numpy(force=True) > 0,
    pred_av.numpy(force=True) > 0,
    show_gt_graph=False,
)
m

GridBox(children=(HTML(value='<h3 style="text-align: center;">Reference Topology: 092_N</h3>'), HTML(value='<h…

In [9]:
STOP

NameError: name 'STOP' is not defined

In [ ]:
np.set_printoptions(linewidth=200, precision=2, suppress=True)
d = out.to_digraph()
d.lines_info(b1=86, sort_by_p=True).round(3).head(20)

,b0,b1,tip0,tip1,line_p,av_p,total_p,b0_dir_p,b1_dir_p
0,16,86,1,0,0.949,0.0,1.451,0.007,0.998
1,129,86,0,1,0.907,0.0,1.366,0.916,0.002
2,38,86,0,0,0.051,0.0,1.047,0.995,0.998
3,-1,86,0,0,0.000,0.0,0.998,0.002,0.998
4,36,86,1,0,0.000,0.0,0.997,0.997,0.998
5,35,86,1,0,0.000,0.0,0.800,0.602,0.998
6,44,86,0,0,0.000,0.0,0.675,0.352,0.998
7,41,86,0,0,0.000,0.0,0.572,0.147,0.998
8,37,86,0,0,0.000,0.0,0.534,0.069,0.998
9,43,86,0,0,0.000,0.0,0.511,0.024,0.998


In [17]:
import tqdm

from fundus_toolkits import AVLabel
from fundus_toolkits.utils.geometric import Point
from fundus_vessels_toolkit.segment_to_graph.av_tree_parsing import naive_infer_arborescence

name = []
pred_parent_acc = []
pred_dir_acc = []
pred_av_acc = []
opti_parent_acc = []
opti_dir_acc = []
opti_av_acc = []
baseline_parent_acc = []
baseline_dir_acc = []
baseline_av_acc = []

eval_set = test_set

with torch.inference_mode():
    for i in tqdm.tqdm(range(len(eval_set))):
        sample, gt_digraph = eval_set.get(i, version="fvt", return_digraph=True)
        assert VBranchDigraph.has_all_p(gt_digraph) and gt_digraph.graph is not None

        valid_branch = ~gt_digraph.branch_fp()
        av_gt = (gt_digraph.branch_av_class() <= 1)[valid_branch]
        od = Point.parse(sample.od_yx.tolist())

        art_branch = gt_digraph.graph.branch_attr["av"] == AVLabel.ART
        vei_branch = gt_digraph.graph.branch_attr["av"] == AVLabel.VEI
        parent_base = -np.ones(gt_digraph.graph.branch_count, dtype=np.int_)
        dir_base = np.zeros(gt_digraph.graph.branch_count, dtype=np.bool_)
        parent_base[art_branch], dir_base[art_branch] = naive_infer_arborescence(
            gt_digraph.graph, od, branch_subset=art_branch
        )
        parent_base[vei_branch], dir_base[vei_branch] = naive_infer_arborescence(
            gt_digraph.graph, od, branch_subset=vei_branch
        )
        parent_base, dir_base = parent_base[valid_branch], dir_base[valid_branch]
        baseline_parent_acc.append((parent_base == gt_digraph.max_parent()[valid_branch]).mean())
        baseline_dir_acc.append((dir_base == (gt_digraph.branch_dir_p[valid_branch] > 0.5)).mean())
        baseline_av_acc.append((art_branch[valid_branch] == av_gt).mean())

        out = model(sample.cuda())
        name += [out.name]
        pred_parent, pred_dir = out.max_parent(), out.dir_logit > 0

        pred_parent, pred_dir = pred_parent.numpy(force=True), pred_dir.numpy(force=True)
        pred_parent, pred_dir = pred_parent[valid_branch], pred_dir[valid_branch]
        av_logit = out.av_logit.numpy(force=True)[valid_branch]
        pred_parent_acc.append((pred_parent == gt_digraph.max_parent()[valid_branch]).mean())
        pred_dir_acc.append((pred_dir == (gt_digraph.branch_dir_p[valid_branch] > 0.5)).mean())
        pred_av_acc.append(((av_logit > 0) == av_gt).mean())

        opti_parent, opti_dir, opti_av = out.optimal_tree
        opti_parent, opti_dir = opti_parent.numpy(force=True), opti_dir.numpy(force=True)
        opti_parent, opti_dir = opti_parent[valid_branch], opti_dir[valid_branch]
        opti_parent_acc.append((opti_parent == gt_digraph.max_parent()[valid_branch]).mean())
        opti_dir_acc.append((opti_dir == (gt_digraph.branch_dir_p[valid_branch] > 0.5)).mean())

        opti_av = opti_av.numpy(force=True)[valid_branch]
        opti_av_acc.append(((opti_av > 0) == av_gt).mean())

  0%|          | 0/73 [00:00<?, ?it/s]

100%|██████████| 73/73 [00:40<00:00,  1.79it/s]


Parent ACC


In [18]:
np.mean(pred_parent_acc), np.mean(opti_parent_acc), np.mean(baseline_parent_acc)

(np.float64(0.8906831458820422),
 np.float64(0.8889041721575117),
 np.float64(0.8064986038845119))

In [9]:
np.mean(pred_parent_acc), np.mean(opti_parent_acc), np.mean(baseline_parent_acc)

(np.float64(0.9053021935282526),
 np.float64(0.9033054015663307),
 np.float64(0.8261209401315641))

In [ ]:
np.mean(pred_parent_acc), np.mean(opti_parent_acc), np.mean(baseline_parent_acc)

(np.float64(0.8606960002811269),
 np.float64(0.8733594971206884),
 np.float64(0.8261209401315641))

In [21]:
np.argsort(pred_parent_acc)

array([58, 60, 59, 72, 41, 55, 64, 16, 23, 52, 10, 40, 35, 12, 50, 15, 25,
       37, 49,  0, 39,  4, 48,  1, 27, 70, 61, 45, 18, 36, 31, 24, 19, 22,
       38, 28,  2, 62,  9, 67, 68, 71, 42, 66, 57, 51,  5, 33, 46, 32, 47,
       56,  3, 63, 21, 34, 43,  8, 54, 29, 20, 69, 44, 26, 65,  7, 17, 30,
        6, 11, 13, 14, 53])

Dir ACC


In [19]:
np.mean(pred_dir_acc), np.mean(opti_dir_acc), np.mean(baseline_dir_acc)

(np.float64(0.989930563287593),
 np.float64(0.9897025119500504),
 np.float64(0.9263301467902944))

In [10]:
np.mean(pred_dir_acc), np.mean(opti_dir_acc), np.mean(baseline_dir_acc)

(np.float64(0.9895811311040469),
 np.float64(0.9890553388113293),
 np.float64(0.9359552166526669))

In [ ]:
np.mean(pred_dir_acc), np.mean(opti_dir_acc), np.mean(baseline_dir_acc)

(np.float64(0.9677232103487156),
 np.float64(0.9765069920258027),
 np.float64(0.9359552166526669))

AV ACC


In [20]:
np.mean(pred_av_acc), np.mean(opti_av_acc), np.mean(baseline_av_acc)

(np.float64(0.9159254171099341),
 np.float64(0.8751948123913513),
 np.float64(0.9335695582648449))

In [11]:
np.mean(pred_av_acc), np.mean(opti_av_acc), np.mean(baseline_av_acc)

(np.float64(0.9310911063162475),
 np.float64(0.8950026974273463),
 np.float64(0.949384537052953))

In [ ]:
np.mean(pred_av_acc), np.mean(opti_av_acc), np.mean(baseline_av_acc)

(np.float64(0.9112861514322247),
 np.float64(0.880420397243731),
 np.float64(0.949384537052953))

In [43]:
np.argsort(pred_av_acc)

array([61, 60, 24, 58, 15, 36, 59,  3, 25, 18, 72, 42,  9, 12, 70, 64, 23,
       31, 33,  8,  0, 40, 38, 27, 55, 21, 16, 41, 10,  7, 52,  1, 37, 45,
       50,  2, 35, 34, 71, 49, 22, 48, 57, 39,  4, 51, 13, 47, 28, 62, 17,
       44, 29, 56, 66,  6, 53, 68, 43, 20,  5, 19, 32, 67, 63, 54, 26, 14,
       69, 30, 11, 65, 46])